## ANN Classifier

In [7]:
import torch
from typing import Tuple, Dict, Any, List
import torch.nn.functional as F
import numpy as np
import time
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class ANNClassifier:
    def __init__(self, input_dim=None, hidden_dim1=224, hidden_dim2=128, C=1.0, max_iter=100, verbose=True, random_state=42):
        self.C = C
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim  # Will be set during fit() if None
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.model = None

    def compute_loss(self, preds, labels):
        loss = self.loss_func(preds, labels)
        wreg = 0.5 * sum((param.norm(p=2) for param in self.model.parameters()))  # L2 regularization
        return loss.mean() + (1.0 / self.C) * wreg

    def predict_proba(self, feats):
        feats = feats.to(self.device)
        with torch.no_grad():
            return torch.softmax(self.model(feats), dim=1)

    def fit(self, feats, labels):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        # Set input_dim if not set
        if self.input_dim is None:
            self.input_dim = feats.shape[1]

        self.model = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, self.hidden_dim1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim1, self.hidden_dim2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim2, 9)  # Assuming 9 classes
        ).to(self.device)

        for layer in self.model:
            if isinstance(layer, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(layer.weight)

        feats = feats.to(self.device)
        labels = labels.long().to(self.device)

        opt = torch.optim.Adam(self.model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)

        min_loss = np.inf
        patience = 10
        patience_counter = 0

        for epoch in range(self.max_iter):
            opt.zero_grad()
            preds = self.model(feats)
            loss = self.compute_loss(preds, labels)
            loss.backward()
            opt.step()
            scheduler.step(loss)

            if self.verbose and epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss: {loss:.4f}")

            if loss < min_loss:
                min_loss = loss
                patience_counter = 0
                best_model = self.model.state_dict()
            else:
                patience_counter += 1

            # if patience_counter >= patience:
            #     print(f"Early stopping at epoch {epoch}")
            #     break

        self.model.load_state_dict(best_model)

        if self.verbose:
            print(f"Final Loss: {min_loss:.4f}")

In [8]:
def test_ANN_probe(
    classifier: ANNClassifier,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    prefix: str = "ann_",
    verbose: bool = True,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    if verbose:
        print(f"ANN Probe Evaluation: Test shape {test_feats.shape}")

    probs_all = classifier.predict_proba(test_feats).cpu().numpy()
    preds_all = np.argmax(probs_all, axis=1)
    targets_all = test_labels.cpu().numpy()
    roc_kwargs = {"multi_class": "ovo", "average": "macro"}

    # ⬇️ Calculate accuracy
    accuracy = np.mean(preds_all == targets_all)
    print(f"✅ Test Accuracy: {accuracy * 100:.2f}%")

    # ⬇️ Eval metrics
    eval_metrics = get_eval_metrics(targets_all, preds_all, probs_all, True, prefix, roc_kwargs)
    dump = {"preds_all": preds_all, "probs_all": probs_all, "targets_all": targets_all}

    # ⬇️ Confusion matrix
    conf_matrix = confusion_matrix(targets_all, preds_all)
    print("Confusion Matrix:")
    plot_confusion_matrix(conf_matrix)
    
    # ⬇️ New plots for deeper insight
    plot_classification_report(targets_all, preds_all)
    plot_roc_multiclass(targets_all, probs_all, n_classes=9)
    plot_prediction_confidence(probs_all, preds_all, targets_all)


    return eval_metrics, dump


In [9]:
# Do this in main
import pandas as pd

df= pd.read_csv(r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv")
valid_patch_names = set(df['patch_name'].astype(str))

def is_non_white_patch(patch_path: str, valid_patch_names: set) -> int:
    patch_name = os.path.basename(patch_path)
    return int(patch_name in valid_patch_names)

def extract_coords(patch_name: str):
    try:
        base = os.path.splitext(patch_name)[0]
        parts = base.split('_')
        x = int([p for p in parts if p.startswith('x')][0][1:])
        y = int([p for p in parts if p.startswith('y')][0][1:])
        return x, y
    except Exception as e:
        print(f"❌ Error extracting coords from {patch_name}: {e}")
        return None, None

## Conch 512 patch and crc100k 512 size

In [10]:
import json

with open(r'D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\best_parameters\best_params_upscale_conch.json', 'r') as f:
    best_params = json.load(f)

print("✅ Loaded best_params from best_params_uni2.json",best_params)

✅ Loaded best_params from best_params_uni2.json {'hidden_dim1': 224, 'hidden_dim2': 128, 'C': 10.0, 'max_iter': 500}


### conch features 512 pipeline

In [11]:
model_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\models\ann_crc100k_upscale_conch15.pth"

In [12]:
input_dim = 768

# Initialize classifier
model = ANNClassifier(
    input_dim=input_dim,
    hidden_dim1=best_params['hidden_dim1'],
    hidden_dim2=best_params['hidden_dim2'],
    C=best_params['C'],
    max_iter=best_params.get('max_iter', 100),
    verbose=True
)

# Build model architecture (required before loading weights)
model.model = torch.nn.Sequential(
    torch.nn.Linear(model.input_dim, model.hidden_dim1),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim1, model.hidden_dim2),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim2, 9)  # 9 classes assumed
).to(model.device)

# Load trained weights
model.model = torch.load(model_path, map_location=model.device)
model.model.to(model.device)
model.model.eval()

Sequential(
  (0): Linear(in_features=768, out_features=224, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=224, out_features=128, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=128, out_features=9, bias=True)
)

## New approach patch label

In [13]:
import os
import h5py
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ========== CONFIGURATION ==========

patches_root = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data"  # Folder containing slide subfolders with PNG patches
features_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5"  # Folder containing slide subfolders with patch_features\*.h5
non_white_csv = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv"
patch_groups_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Patch_groups_dict\conch_512_512"
output_slide_features = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\conch_512_512"
BATCH_SIZE = 64  # You can adjust this based on your GPU memory
# ===================================

# Load valid patch names
non_white_df = pd.read_csv(non_white_csv)
valid_patch_names = set(non_white_df['patch_name'].astype(str))

# Collect all updates here for later update
updates = []

# Main loop
for slide_name in tqdm(os.listdir(patches_root), desc="🔍 Processing slides"):
    slide_patch_folder = os.path.join(patches_root, slide_name)
    if not os.path.isdir(slide_patch_folder):
        continue

    h5_path = os.path.join(features_root, slide_name, "patch_features", f"{slide_name}_patch_features.h5")
    if not os.path.exists(h5_path):
        print(f"⚠️ H5 file not found for {slide_name}")
        continue

    try:
        with h5py.File(h5_path, "r") as h5_file:
            coords = h5_file['coords'][:].squeeze(0)
            features = h5_file['features'][:].squeeze(0)
            coord_to_feat = {tuple(c): f for c, f in zip(coords, features)}

            batch_features = []
            batch_patch_names = []

            patch_files = os.listdir(slide_patch_folder)
            for idx, patch_file in enumerate(patch_files):
                patch_path = os.path.join(slide_patch_folder, patch_file)

                if is_non_white_patch(patch_path, valid_patch_names) == 0:
                    continue

                x, y = extract_coords(patch_file)
                if (x, y) in coord_to_feat:
                    current_feature = coord_to_feat[(x, y)]
                    batch_features.append(current_feature)
                    batch_patch_names.append(patch_file)

                    last_patch = (idx == len(patch_files) - 1)

                    if len(batch_features) == BATCH_SIZE or last_patch:
                        tensor_batch = torch.tensor(batch_features, dtype=torch.float32).to(model.device)
                        model.model.eval()
                        with torch.no_grad():
                            outputs = model.model(tensor_batch)
                        predictions = torch.argmax(outputs, dim=1).cpu().numpy()

                        for patch_name, pred in zip(batch_patch_names, predictions):
                            updates.append((patch_name, pred))

                        batch_features = []
                        batch_patch_names = []
                else:
                    print(f"⚠️ Coordinates ({x}, {y}) not found in h5: {patch_file}")

    except Exception as e:
        print(f"❌ Error processing {h5_path}: {e}")


🔍 Processing slides: 100%|██████████████████████████████████████████████████████████| 417/417 [03:02<00:00,  2.29it/s]


In [ ]:
import os
import torch
from tqdm import tqdm

# Main loop
for slide_name in tqdm(os.listdir(patches_root), desc="🔍 Processing slides"):
    slide_patch_folder = os.path.join(patches_root, slide_name)

    if not os.path.isdir(slide_patch_folder):
        continue

    patch_files = sorted(os.listdir(slide_patch_folder))  # Ensure consistency

    batch_features = []
    batch_patch_names = []

    for idx, patch_file in enumerate(patch_files):
        patch_path = os.path.join(slide_patch_folder, patch_file)

        if is_non_white_patch(patch_path, valid_patch_names) == 0:
            continue

        try:
            features = torch.load(patch_path)  # Shape should be (5, 768)
            if not isinstance(features, torch.Tensor):
                raise ValueError(f"Expected tensor, got {type(features)}")

            if features.shape != (5, 768):
                print(f"⚠️ Unexpected feature shape in {patch_file}: {features.shape}")
                continue

            # Split into 5 tensors of shape (1, 768)
            for i in range(5):
                single_vector = features[i].unsqueeze(0)  # shape: (1, 768)
                batch_features.append(single_vector.squeeze(0).numpy())

                # Create modified patch name with suffix
                base_name = os.path.splitext(patch_file)[0]  # remove .pt
                suffix = f"_{i+1}.png"
                short_name = "_".join(base_name.split("_")[1:])  # remove 'TCGA-3L' etc. prefix
                new_name = f"{short_name}{suffix}"
                batch_patch_names.append(new_name)

        except Exception as e:
            print(f"❌ Error reading {patch_path}: {e}")
            continue

        # Either batch is full or it's the last patch
        last_patch = (idx == len(patch_files) - 1)
        if len(batch_features) >= BATCH_SIZE or last_patch:
            tensor_batch = torch.tensor(batch_features, dtype=torch.float32).to(model.device)
            model.model.eval()
            with torch.no_grad():
                outputs = model.model(tensor_batch)
            predictions = torch.argmax(outputs, dim=1).cpu().numpy()

            for patch_name, pred in zip(batch_patch_names, predictions):
                updates.append((patch_name, pred))

            # Reset for next batch
            batch_features = []
            batch_patch_names = []


In [14]:
update_df = pd.DataFrame(updates, columns=['patch_name', 'tissue_label'])
non_white_df.set_index('patch_name', inplace=True)
non_white_df.update(update_df.set_index('patch_name'))
non_white_df.reset_index(inplace=True)
non_white_df = non_white_df.merge(update_df, on='patch_name', how='left')

# Save the updated CSV if needed
non_white_df.to_csv("conch_upscaled_labeled_df.csv", index=False)

In [15]:
non_white_df

,patch_name,patch_location,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,tissue_label
0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0.0
1,TCGA-3L-AA1B_nonMSIH_x10288_y13024_patch00160.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,160,10288,13024,233.85,218.41,224.88,24.24,38.86,31.59,0,0.0
2,TCGA-3L-AA1B_nonMSIH_x10288_y13536_patch00161.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,161,10288,13536,190.05,164.94,186.11,65.08,74.05,58.34,0,0.0
3,TCGA-3L-AA1B_nonMSIH_x10288_y14048_patch00162.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,162,10288,14048,156.51,107.05,145.28,60.21,52.50,46.58,0,3.0
4,TCGA-3L-AA1B_nonMSIH_x10288_y14560_patch00163.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,163,10288,14560,196.97,115.28,154.36,37.78,33.66,29.73,0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377928,TCGA-WS-AB45_MSIH_x54208_y23040_patch04527.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,4527,54208,23040,212.47,169.26,195.15,31.72,50.20,35.92,0,4.0
1377929,TCGA-WS-AB45_MSIH_x54208_y23552_patch04528.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,4528,54208,23552,214.06,168.04,193.17,33.29,53.89,38.92,0,4.0
1377930,TCGA-WS-AB45_MSIH_x54208_y24064_patch04529.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,4529,54208,24064,217.35,185.47,204.79,34.86,54.06,38.92,0,7.0
1377931,TCGA-WS-AB45_MSIH_x54208_y24576_patch04530.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,4530,54208,24576,223.91,202.11,215.23,31.18,51.20,36.45,0,5.0


## Aggregation

In [18]:
# non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
import re
non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
non_white_df['patch_name'] = non_white_df['patch_name'].str.replace(r'_patch\d{5}', '', regex=True)

In [19]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

root_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_patchbypatch"
saving_path = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\conch_512_512"
data = []

# Iterate through each subfolder with tqdm
for subfolder_name in tqdm(os.listdir(root_dir), desc="Processing slides"):
    subfolder_path = os.path.join(root_dir, subfolder_name)

    if not os.path.isdir(subfolder_path):
        continue

    data = []  # clear for each slide

    # Loop through .pt files in this subfolder
    for file_name in os.listdir(subfolder_path):
        if file_name.endswith(".pt"):
            pt_path = os.path.join(subfolder_path, file_name)

            try:
                tensor = torch.load(pt_path)

                if isinstance(tensor, torch.Tensor):
                    flat_feat = tensor.view(-1).tolist()
                else:
                    print(f"Unexpected type in {file_name}: {type(tensor)}")
                    flat_feat = []

                data.append({
                    "patch_name": file_name,
                    "features": flat_feat
                })

            except Exception as e:
                print(f"Error loading {pt_path}: {e}")

    # Skip if no patches found
    if not data:
        continue

    df = pd.DataFrame(data)
    df = df.merge(non_white_df[['patch_name', 'tissue_label']], on='patch_name', how='left')

    num_classes = 9
    feature_dim = len(df['features'].iloc[0])

    group_features = []

    for label in range(num_classes):
        group_df = df[df['tissue_label'] == label]

        if not group_df.empty:
            features = np.stack(group_df['features'].to_numpy())
            group_avg = np.mean(features, axis=0)
        else:
            group_avg = np.zeros(feature_dim)

        group_features.append(group_avg)

    final_array = np.stack(group_features, axis=0)
    final_tensor = torch.tensor(final_array, dtype=torch.float32)

    # Save the tensor
    save_path = os.path.join(saving_path, f"{subfolder_name}.pt")
    torch.save(final_tensor, save_path)

Processing slides: 100%|███████████████████████████████████████████████████████████| 417/417 [1:35:38<00:00, 13.76s/it]


## Previos approach

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle

# ========== CONFIGURATION ==========
patches_root = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data"  # Folder containing slide subfolders with PNG patches
features_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5"  # Folder containing slide subfolders with patch_features\*.h5
non_white_csv = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv"
patch_groups_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Patch_groups_dict\conch_512_512"
output_slide_features = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\conch_512_512"
os.makedirs(patch_groups_dir, exist_ok=True)

# ===================================

# Load valid patch names
non_white_df = pd.read_csv(non_white_csv)
valid_patch_names = set(non_white_df['patch_name'].astype(str))
count=0
num_groups = 9
feature_dim = 768


counter=0
# Main loop
for slide_name in tqdm(os.listdir(patches_root), desc="🔍 Processing slides"):
    slide_patch_folder = os.path.join(patches_root, slide_name) #'D:\\Aamir Gulzar\\KSA_project2\\dataset\\patch_data\\TCGA-5M-AATE_nonMSIH'
    patch_group_path = os.path.join(patch_groups_dir, f"{slide_name}.pkl")
    print(f"processing slide {slide_name}")
    if not os.path.isdir(slide_patch_folder):
        continue

    h5_path = os.path.join(features_root, slide_name, "patch_features", f"{slide_name}_patch_features.h5")
    if not os.path.exists(h5_path):
        print(f"⚠️ H5 file not found for {slide_name}")
        continue

    try:
        with h5py.File(h5_path, "r") as h5_file:
            coords = h5_file['coords'][:].squeeze(0)    # (N, 2)
            features = h5_file['features'][:].squeeze(0)  # (N, D)

            coord_to_feat = {tuple(c): f for c, f in zip(coords, features)}
            
            patch_group_dict = {i: [] for i in range(num_groups)}

            for patch_file in os.listdir(slide_patch_folder): #'D:\\Aamir Gulzar\\KSA_project2\\dataset\\patch_data\\TCGA-5M-AATE_nonMSIH'
                patch_path = os.path.join(slide_patch_folder, patch_file) #'TCGA-5M-AAT6_MSIH_x9696_y31088_patch00277.png'

                if is_non_white_patch(patch_path, valid_patch_names) == 0:
                    continue
                
                x, y = extract_coords(patch_file)
                if (x, y) in coord_to_feat:
                    current_feature = coord_to_feat[(x, y)]
                    # current_feature = np.expand_dims(current_feature, axis=0)  # Now shape is (1, 768)

                    # === Your own processing logic here ===
                    # print(f"🧪 Processing patch: {patch_file}, Feature vector shape: {current_feature.shape}")
                   # Move feature to the same device as model
                    current_feature = torch.tensor(current_feature, dtype=torch.float32).unsqueeze(0).to(model.device)
                    
                    # Run inference
                    model.model.eval()
                    with torch.no_grad():
                        output = model.model(current_feature)
                        predicted_class = torch.argmax(output, dim=1).item()
                    
                    # print("Predicted class label:", predicted_class)

                    non_white_df.loc[non_white_df['patch_name'] == patch_file, 'tissue_label'] = predicted_class
                    patch_group_dict[predicted_class].append(current_feature)

                else:
                    print(f"⚠️ Coordinates ({x}, {y}) not found in h5: {patch_file}")
            
            # Average features in each group            
            group_features = []
            
            for group_id in range(num_groups):
                feats = patch_group_dict[group_id]
                if len(feats) > 0:
                    feats_np = [f.cpu().numpy() for f in feats]
                    group_avg = np.mean(feats_np, axis=0)
                else:
                    group_avg = np.zeros(feature_dim)
                
                group_features.append(group_avg)


            # Final slide-level feature vector
            # This is basically the patch level features but averaged with its whole group, this is the aggregation method.
            slide_vector = np.stack(group_features, axis=0)  # shape: (9, 768)
            slide_tensor = torch.from_numpy(slide_vector)

            # Create directory for this slide
            slide_folder = os.path.join(output_slide_features, slide_name)
            os.makedirs(slide_folder, exist_ok=True)

            # Save as pt in folder
            torch.save(slide_tensor, os.path.join(slide_folder, f"{slide_name}.pt"))

            # Save patch_group_dict as .pkl
            with open(patch_group_path, 'wb') as f:
                pickle.dump(patch_group_dict, f)

    except Exception as e:
        print(f"❌ Error processing {h5_path}: {e}")

🔍 Processing slides:   0%|                                                                    | 0/417 [00:00<?, ?it/s]

🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png, Feature vector shape: (768,)
Predicted class label: 0
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y13024_patch00160.png, Feature vector shape: (768,)
Predicted class label: 0
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y13536_patch00161.png, Feature vector shape: (768,)
Predicted class label: 0
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y14048_patch00162.png, Feature vector shape: (768,)
Predicted class label: 3
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y14560_patch00163.png, Feature vector shape: (768,)
Predicted class label: 5
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y15072_patch00164.png, Feature vector shape: (768,)
Predicted class label: 5
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y15584_patch00165.png, Feature vector shape: (768,)
Predicted class label: 5
🧪 Processing patch: TCGA-3L-AA1B_nonMSIH_x10288_y16096_patch00166.png, Feature vector shape: (768,)
Predicted class label: 5


🔍 Processing slides:   0%|▏                                                       | 1/417 [03:01<20:58:04, 181.45s/it]

🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x0_y46528_patch00000.png, Feature vector shape: (768,)
Predicted class label: 1
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10240_y46528_patch00040.png, Feature vector shape: (768,)
Predicted class label: 1
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x1024_y46528_patch00004.png, Feature vector shape: (768,)
Predicted class label: 1
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10432_y13856_patch01003.png, Feature vector shape: (768,)
Predicted class label: 0
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10432_y14368_patch01004.png, Feature vector shape: (768,)
Predicted class label: 0
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10432_y14880_patch01005.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10432_y15392_patch01006.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-4N-A93T_nonMSIH_x10432_y15904_patch01007.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Pro

🔍 Processing slides:   0%|▎                                                       | 2/417 [07:50<28:12:54, 244.76s/it]

...

🧪 Processing patch: TCGA-A6-5666_nonMSIH_x9968_y9552_patch01668.png, Feature vector shape: (768,)
Predicted class label: 8
❌ Error processing D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5\TCGA-A6-5666_nonMSIH\patch_features\TCGA-A6-5666_nonMSIH_patch_features.h5: all input arrays must have the same shape
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y2384_patch00199.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y2896_patch00200.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y3408_patch00201.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y3920_patch00202.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y4432_patch00203.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-5667_nonMSIH_x1120_y4944_patch00204.png,

🔍 Processing slides:   5%|██▋                                                      | 20/417 [51:24<8:18:25, 75.33s/it]

❌ Error processing D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5\TCGA-A6-5667_nonMSIH\patch_features\TCGA-A6-5667_nonMSIH_patch_features.h5: all input arrays must have the same shape
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y2048_patch01088.png, Feature vector shape: (768,)
Predicted class label: 6
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y2560_patch01089.png, Feature vector shape: (768,)
Predicted class label: 6
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y3072_patch01090.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y3584_patch01091.png, Feature vector shape: (768,)
Predicted class label: 6
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y4096_patch01092.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y4608_patch01093.png, Feature vector shape: (768,)
Predicted class label: 6
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x1000_y5120_patch01094.png,

🔍 Processing slides:   5%|██▊                                                      | 21/417 [53:11<9:20:44, 84.96s/it]

🧪 Processing patch: TCGA-A6-6137_nonMSIH_x9704_y9216_patch01458.png, Feature vector shape: (768,)
Predicted class label: 8
🧪 Processing patch: TCGA-A6-6137_nonMSIH_x9704_y9728_patch01459.png, Feature vector shape: (768,)
Predicted class label: 8
❌ Error processing D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5\TCGA-A6-6137_nonMSIH\patch_features\TCGA-A6-6137_nonMSIH_patch_features.h5: all input arrays must have the same shape
🧪 Processing patch: TCGA-A6-6138_nonMSIH_x10464_y10040_patch00164.png, Feature vector shape: (768,)
Predicted class label: 3
🧪 Processing patch: TCGA-A6-6138_nonMSIH_x10464_y10552_patch00165.png, Feature vector shape: (768,)
Predicted class label: 3
🧪 Processing patch: TCGA-A6-6138_nonMSIH_x10464_y4920_patch00154.png, Feature vector shape: (768,)
Predicted class label: 5
🧪 Processing patch: TCGA-A6-6138_nonMSIH_x10464_y5432_patch00155.png, Feature vector shape: (768,)
Predicted class label: 5
🧪 Processing patch: TCGA-A6-6138_nonMSIH_x10464_y5944_patch001

In [ ]:
non_white_df.to_csv("conch_upscaled_labeled_df.csv", index=False)